In [ ]:
# -*- coding: utf-8 -*-
"""
AUTOENCODER TÉRMICO SAUDÁVEL — REMOVE TEMPERATURA SEM COPIAR A REFERÊNCIA

Uso correto do autoencoder neste problema:
1) O AE é treinado SOMENTE com curvas saudáveis, falha = 0.
2) O AE aprende a estimar a curva saudável equivalente à temperatura da amostra: H_hat_T.
3) A curva compensada NÃO é a saída direta do AE.
4) A compensação final preserva o resíduo da curva original:

       X_comp = X_original + ALPHA_COMP * (H_REF - H_hat_T)

   ou, equivalentemente:

       X_comp - H_REF ≈ X_original - H_hat_T

Assim, o AE remove a componente térmica saudável estimada, mas não tenta reconstruir o dano.
"""

# ============================================================
# 1) IMPORTS
# ============================================================

import os
import re
import time
import copy
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore", category=UserWarning)


# ============================================================
# 2) PARÂMETROS
# ============================================================

ARQ_BASE = "base-completo--.pkl"

REF_TEMP = 30

FREQ_MIN_KHZ = 40
FREQ_MAX_KHZ = 50

OUTPUT_DIR = f"AE_TERMICO_PRESERVA_DANO_{REF_TEMP}C_{FREQ_MIN_KHZ}-{FREQ_MAX_KHZ}kHz"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Autoencoder
EPOCHS = 500
BATCH_SIZE = 8
LR = 5e-4
PATIENCE = 90
LATENT_DIM = 8
DROPOUT = 0.15
INPUT_NOISE_STD = 0.025
NUM_WORKERS = 0

# Suavização da saída do AE.
# Isso ajuda a saída do AE representar H_T, e não detalhes locais de dano.
# Use 1 para desativar.
AE_OUTPUT_SMOOTH_KERNEL = 7

# Quanto da correção térmica será aplicada.
# Se estiver mexendo demais, teste 0.70 ou 0.80.
# Se estiver compensando pouco, teste 1.00.
ALPHA_COMP = 0.90

# Limite da correção térmica em unidades de desvio padrão saudável por ponto.
# Evita correções absurdas.
DELTA_CLIP_STD = 3.0

# Pesos da loss
LAMBDA_RECON = 1.00
LAMBDA_DERIV = 0.40
LAMBDA_CURVATURE = 0.20
LAMBDA_CORR = 0.05
LAMBDA_TEMP = 0.10

# Figuras
TEMP_ESCOLHIDA = 55
DANOS_PLOTAR = [0, 1, 2]
OCORRENCIA_CURVA = 0
HIST_BINS = 18

np.random.seed(42)
torch.manual_seed(42)


# ============================================================
# 3) FUNÇÕES GERAIS
# ============================================================

def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None


def get_freq_columns(df, fmin_khz, fmax_khz):
    cols = []
    freqs = []

    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            f_khz = f / 1e3
            if fmin_khz <= f_khz <= fmax_khz:
                cols.append(c)
                freqs.append(f)

    order = np.argsort(freqs)
    fcols = [cols[i] for i in order]
    fhz = np.array(freqs, dtype=float)[order]
    return fcols, fhz


def aplicar_estilo_artigo():
    plt.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": 18,
        "axes.labelsize": 20,
        "axes.titlesize": 20,
        "xtick.labelsize": 17,
        "ytick.labelsize": 17,
        "legend.fontsize": 14,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "pdf.fonttype": 42,
        "ps.fonttype": 42
    })


def salvar_figura(fig, nome_base, output_dir=OUTPUT_DIR, dpi=600):
    os.makedirs(output_dir, exist_ok=True)
    png_path = os.path.join(output_dir, f"{nome_base}.png")
    pdf_path = os.path.join(output_dir, f"{nome_base}.pdf")
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    print(f"Figura salva em PNG: {png_path}")
    print(f"Figura salva em PDF: {pdf_path}")
    return png_path, pdf_path


def formatar_temp(T):
    if float(T).is_integer():
        return f"{int(T)}"
    return f"{T:.1f}"


# ============================================================
# 4) REFERÊNCIAS SAUDÁVEIS
# ============================================================

def get_healthy_references_by_temperature(df, fcols, ref_temp):
    if "falha" not in df.columns:
        raise ValueError("O DataFrame precisa ter a coluna 'falha'.")
    if "temperatura_c" not in df.columns:
        raise ValueError("O DataFrame precisa ter a coluna 'temperatura_c'.")

    df_h = df[df["falha"] == 0].copy()
    if len(df_h) == 0:
        raise ValueError("Não há curvas saudáveis, isto é, falha = 0.")

    temps_h = np.array(sorted(df_h["temperatura_c"].unique()), dtype=float)
    healthy_by_temp = {}

    for T in temps_h:
        pool = df_h.loc[np.isclose(df_h["temperatura_c"], T), fcols].to_numpy(float)
        healthy_by_temp[float(T)] = np.median(pool, axis=0)

    ref_temp_used = float(temps_h[np.argmin(np.abs(temps_h - ref_temp))])
    if not np.isclose(ref_temp_used, ref_temp):
        print(f"AVISO: não existe saudável exatamente em {ref_temp} °C. Usando {ref_temp_used} °C.")

    y_ref_healthy = healthy_by_temp[ref_temp_used]
    return healthy_by_temp, temps_h, y_ref_healthy, ref_temp_used


def get_nearest_healthy_curve(healthy_by_temp, healthy_temps, temperatura):
    healthy_temps = np.asarray(healthy_temps, dtype=float)
    temp_used = float(healthy_temps[np.argmin(np.abs(healthy_temps - temperatura))])
    return healthy_by_temp[temp_used], temp_used


# ============================================================
# 5) MÉTRICAS
# ============================================================

def rmsd(y, ref):
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)
    return float(np.sqrt(np.mean((y - ref) ** 2)))


def ccdm(y, ref):
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)
    y0 = y - np.mean(y)
    r0 = ref - np.mean(ref)
    num = float(np.sum(y0 * r0))
    den = float(np.sqrt(np.sum(y0 ** 2) * np.sum(r0 ** 2)) + 1e-18)
    return float(1 - num / den)


def corr_loss_batch(y_pred, y_true):
    yp = y_pred.squeeze(1)
    yt = y_true.squeeze(1)
    yp0 = yp - yp.mean(dim=1, keepdim=True)
    yt0 = yt - yt.mean(dim=1, keepdim=True)
    num = torch.sum(yp0 * yt0, dim=1)
    den = torch.sqrt(torch.sum(yp0 ** 2, dim=1) * torch.sum(yt0 ** 2, dim=1) + 1e-12)
    return torch.mean(1.0 - num / den)


def derivative_loss(y_pred, y_true):
    dy_pred = y_pred[:, :, 1:] - y_pred[:, :, :-1]
    dy_true = y_true[:, :, 1:] - y_true[:, :, :-1]
    return F.smooth_l1_loss(dy_pred, dy_true)


def curvature_loss(y_pred, y_true):
    d2_pred = y_pred[:, :, 2:] - 2 * y_pred[:, :, 1:-1] + y_pred[:, :, :-2]
    d2_true = y_true[:, :, 2:] - 2 * y_true[:, :, 1:-1] + y_true[:, :, :-2]
    return F.smooth_l1_loss(d2_pred, d2_true)


def calcular_metricas_com_preservacao(df_curvas, df_original, fcols, y_ref_healthy,
                                      healthy_by_temp, healthy_temps, metodo):
    X_comp = df_curvas[fcols].to_numpy(float)
    X_orig = df_original[fcols].to_numpy(float)

    df_out = df_curvas.copy()
    rmsd_list = []
    ccdm_list = []
    damage_res_rmsd = []
    damage_res_ccdm = []
    alteracao_rmsd = []

    for i, (_, row) in enumerate(df_original.iterrows()):
        T = float(row["temperatura_c"])
        h_T, _ = get_nearest_healthy_curve(healthy_by_temp, healthy_temps, T)

        y_comp = X_comp[i]
        y_orig = X_orig[i]

        rmsd_list.append(rmsd(y_comp, y_ref_healthy))
        ccdm_list.append(ccdm(y_comp, y_ref_healthy))

        assinatura_esperada = y_orig - h_T
        assinatura_saida = y_comp - y_ref_healthy

        damage_res_rmsd.append(rmsd(assinatura_saida, assinatura_esperada))
        damage_res_ccdm.append(ccdm(assinatura_saida, assinatura_esperada))
        alteracao_rmsd.append(rmsd(y_comp, y_orig))

    df_out["RMSD"] = rmsd_list
    df_out["CCDM"] = ccdm_list
    df_out["DamageResidual_RMSD"] = damage_res_rmsd
    df_out["DamageResidual_CCDM"] = damage_res_ccdm
    df_out["Alteracao_RMSD"] = alteracao_rmsd
    df_out["Metodo"] = metodo
    return df_out


# ============================================================
# 6) AUTOENCODER TÉRMICO SAUDÁVEL
# ============================================================

class HealthyThermalAutoencoder(nn.Module):
    def __init__(self, n_points, latent_dim=8, dropout=0.15):
        super().__init__()
        self.n_points = int(n_points)
        self.latent_dim = int(latent_dim)

        self.encoder = nn.Sequential(
            nn.Linear(self.n_points, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.GELU(),

            nn.Linear(64, self.latent_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(self.latent_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),

            nn.Linear(64, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(128, 256),
            nn.LayerNorm(256),
            nn.GELU(),

            nn.Linear(256, self.n_points)
        )

        # Cabeça auxiliar: ajuda o latente a organizar variação térmica.
        # A temperatura NÃO entra como input; é apenas alvo auxiliar.
        self.temp_head = nn.Sequential(
            nn.Linear(self.latent_dim, 32),
            nn.GELU(),
            nn.Linear(32, 1)
        )

    def smooth_output(self, y):
        if AE_OUTPUT_SMOOTH_KERNEL is None or AE_OUTPUT_SMOOTH_KERNEL <= 1:
            return y
        k = int(AE_OUTPUT_SMOOTH_KERNEL)
        if k % 2 == 0:
            k += 1
        y1 = y.unsqueeze(1)
        y_s = F.avg_pool1d(y1, kernel_size=k, stride=1, padding=k // 2)
        return y_s.squeeze(1)

    def forward(self, x):
        z = self.encoder(x)
        y = self.decoder(z)
        y = self.smooth_output(y)
        t_pred = self.temp_head(z)
        return y, z, t_pred


def preparar_dados_autoencoder_saudavel(df, fcols, y_ref_healthy):
    X_all = df[fcols].to_numpy(float)
    falha = df["falha"].to_numpy()
    temp = df["temperatura_c"].to_numpy(float)

    idx_healthy = np.where(falha == 0)[0]
    if len(idx_healthy) < 5:
        raise ValueError("Poucas curvas saudáveis para treinar o autoencoder.")

    X_healthy = X_all[idx_healthy]
    T_healthy = temp[idx_healthy]

    scaler = StandardScaler()
    scaler.fit(np.vstack([X_healthy, y_ref_healthy.reshape(1, -1)]))

    X_all_s = scaler.transform(X_all)
    X_healthy_s = scaler.transform(X_healthy)
    T_aux = ((T_healthy - REF_TEMP) / 100.0).reshape(-1, 1)

    return {
        "X_all": X_all,
        "X_all_s": X_all_s,
        "X_healthy": X_healthy,
        "X_healthy_s": X_healthy_s,
        "T_healthy": T_healthy,
        "T_aux": T_aux,
        "idx_healthy": idx_healthy,
        "scaler": scaler
    }


def train_autoencoder_saudavel(df, fcols, y_ref_healthy):
    print("\n====================================================")
    print("TREINANDO AUTOENCODER TÉRMICO SAUDÁVEL")
    print("====================================================")
    print("Treino APENAS com curvas saudáveis, falha = 0.")
    print("O AE estima H_T. A curva compensada é calculada fora da rede.")

    dados = preparar_dados_autoencoder_saudavel(df, fcols, y_ref_healthy)
    Xh = dados["X_healthy_s"]
    T_aux = dados["T_aux"]

    idx = np.arange(len(Xh))
    if len(idx) >= 10:
        idx_train, idx_val = train_test_split(idx, test_size=0.20, random_state=42)
    else:
        idx_train = idx
        idx_val = idx

    X_tensor = torch.tensor(Xh, dtype=torch.float32)
    T_tensor = torch.tensor(T_aux, dtype=torch.float32)

    train_dataset = TensorDataset(X_tensor[idx_train], T_tensor[idx_train])
    val_dataset = TensorDataset(X_tensor[idx_val], T_tensor[idx_val])

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Dispositivo usado: {device}")

    model = HealthyThermalAutoencoder(
        n_points=Xh.shape[1],
        latent_dim=LATENT_DIM,
        dropout=DROPOUT
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=40)

    huber = nn.SmoothL1Loss()
    mse = nn.MSELoss()

    best_val = np.inf
    best_state = copy.deepcopy(model.state_dict())
    epochs_sem_melhora = 0

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_recon": [],
        "val_temp": [],
        "lr": []
    }

    for ep in range(1, EPOCHS + 1):
        model.train()
        train_losses = []

        for xb, tb in train_loader:
            xb = xb.to(device)
            tb = tb.to(device)

            opt.zero_grad()

            # Denoising autoencoder:
            # ruído no input, alvo limpo. Isso ajuda a não copiar detalhe local.
            if INPUT_NOISE_STD > 0:
                xb_in = xb + INPUT_NOISE_STD * torch.randn_like(xb)
            else:
                xb_in = xb

            pred, z, tpred = model(xb_in)

            loss_recon = huber(pred, xb)
            loss_deriv = derivative_loss(pred.unsqueeze(1), xb.unsqueeze(1))
            loss_curv = curvature_loss(pred.unsqueeze(1), xb.unsqueeze(1))
            loss_corr = corr_loss_batch(pred.unsqueeze(1), xb.unsqueeze(1))
            loss_temp = mse(tpred, tb)

            loss = (
                LAMBDA_RECON * loss_recon
                + LAMBDA_DERIV * loss_deriv
                + LAMBDA_CURVATURE * loss_curv
                + LAMBDA_CORR * loss_corr
                + LAMBDA_TEMP * loss_temp
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            opt.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        val_recons = []
        val_temps = []

        with torch.no_grad():
            for xb, tb in val_loader:
                xb = xb.to(device)
                tb = tb.to(device)
                pred, z, tpred = model(xb)

                loss_recon = huber(pred, xb)
                loss_deriv = derivative_loss(pred.unsqueeze(1), xb.unsqueeze(1))
                loss_curv = curvature_loss(pred.unsqueeze(1), xb.unsqueeze(1))
                loss_corr = corr_loss_batch(pred.unsqueeze(1), xb.unsqueeze(1))
                loss_temp = mse(tpred, tb)

                loss = (
                    LAMBDA_RECON * loss_recon
                    + LAMBDA_DERIV * loss_deriv
                    + LAMBDA_CURVATURE * loss_curv
                    + LAMBDA_CORR * loss_corr
                    + LAMBDA_TEMP * loss_temp
                )

                val_losses.append(loss.item())
                val_recons.append(loss_recon.item())
                val_temps.append(loss_temp.item())

        train_mean = float(np.mean(train_losses))
        val_mean = float(np.mean(val_losses))
        val_recon = float(np.mean(val_recons))
        val_temp = float(np.mean(val_temps))

        scheduler.step(val_mean)
        current_lr = opt.param_groups[0]["lr"]

        history["epoch"].append(ep)
        history["train_loss"].append(train_mean)
        history["val_loss"].append(val_mean)
        history["val_recon"].append(val_recon)
        history["val_temp"].append(val_temp)
        history["lr"].append(current_lr)

        if val_mean < best_val - 1e-7:
            best_val = val_mean
            best_state = copy.deepcopy(model.state_dict())
            epochs_sem_melhora = 0
        else:
            epochs_sem_melhora += 1

        if ep == 1 or ep % 25 == 0:
            print(
                f"Epoch {ep:4d}/{EPOCHS} | "
                f"train={train_mean:.6f} | val={val_mean:.6f} | "
                f"recon={val_recon:.6f} | temp={val_temp:.6f} | lr={current_lr:.2e}"
            )

        if epochs_sem_melhora >= PATIENCE:
            print(f"Early stopping na epoch {ep}. Melhor val_loss = {best_val:.6f}")
            break

    model.load_state_dict(best_state)

    return {
        "model": model,
        "device": device,
        "scaler": dados["scaler"],
        "history": pd.DataFrame(history),
        "dados": dados,
        "idx_train": idx_train,
        "idx_val": idx_val
    }


def aplicar_autoencoder_compensacao(df, fcols, y_ref_healthy, ae_extra):
    print("\n====================================================")
    print("APLICANDO AUTOENCODER COMO ESTIMADOR DE H_T")
    print("====================================================")
    print("Fórmula: X_comp = X_original + ALPHA_COMP * (H_REF - AE(X_original))")

    model = ae_extra["model"]
    device = ae_extra["device"]
    scaler = ae_extra["scaler"]

    X = df[fcols].to_numpy(float)
    Xs = scaler.transform(X)
    X_tensor = torch.tensor(Xs, dtype=torch.float32)

    model.eval()
    Hhat_scaled_list = []
    latent_list = []
    temp_pred_list = []

    with torch.no_grad():
        for i in range(0, len(X_tensor), BATCH_SIZE):
            xb = X_tensor[i:i + BATCH_SIZE].to(device)
            hhat_s, z, tpred = model(xb)
            Hhat_scaled_list.append(hhat_s.cpu().numpy())
            latent_list.append(z.cpu().numpy())
            temp_pred_list.append(tpred.cpu().numpy())

    Hhat_scaled = np.vstack(Hhat_scaled_list)
    latent = np.vstack(latent_list)
    temp_pred = np.vstack(temp_pred_list)

    Hhat = scaler.inverse_transform(Hhat_scaled)

    # Correção térmica estimada pelo AE.
    delta = y_ref_healthy.reshape(1, -1) - Hhat

    # Limite de segurança baseado na variabilidade das curvas saudáveis.
    Xh = ae_extra["dados"]["X_healthy"]
    std_h = np.std(Xh, axis=0, keepdims=True) + 1e-12
    delta_lim = DELTA_CLIP_STD * std_h
    delta = np.clip(delta, -delta_lim, delta_lim)

    Y_comp = X + ALPHA_COMP * delta

    df_comp = df.copy()
    df_comp[fcols] = Y_comp
    df_comp["temperatura_pred_aux"] = temp_pred[:, 0] * 100.0 + REF_TEMP

    df_hhat = df.copy()
    df_hhat[fcols] = Hhat

    return df_comp, df_hhat, latent


# ============================================================
# 7) GRÁFICOS E RESUMOS
# ============================================================

def resumo_geral(df_long):
    return (
        df_long
        .groupby(["Metodo", "falha"])[[
            "RMSD", "CCDM", "DamageResidual_RMSD", "DamageResidual_CCDM", "Alteracao_RMSD"
        ]]
        .agg(["mean", "std", "min", "max"])
        .round(6)
    )


def resumo_por_temperatura(df_long):
    return (
        df_long
        .groupby(["Metodo", "temperatura_c", "falha"])[[
            "RMSD", "CCDM", "DamageResidual_RMSD", "DamageResidual_CCDM", "Alteracao_RMSD"
        ]]
        .mean()
        .reset_index()
        .sort_values(["Metodo", "temperatura_c", "falha"])
    )


def checar_monotonicidade(df_long, metodos=("Original", "AE térmico")):
    df_use = df_long[df_long["Metodo"].isin(metodos)].copy()
    registros = []

    for metodo in sorted(df_use["Metodo"].unique()):
        df_m = df_use[df_use["Metodo"] == metodo]
        temps = sorted(df_m["temperatura_c"].unique())

        for T in temps:
            df_t = df_m[np.isclose(df_m["temperatura_c"], T)]
            if not {0, 1, 2}.issubset(set(df_t["falha"].unique())):
                continue

            for metrica in ["RMSD", "CCDM"]:
                medias = {}
                for d in [0, 1, 2]:
                    medias[d] = df_t.loc[df_t["falha"] == d, metrica].mean()

                registros.append({
                    "Metodo": metodo,
                    "Temperatura": T,
                    "Metrica": metrica,
                    "D0": medias[0],
                    "D1": medias[1],
                    "D2": medias[2],
                    "Monotonico_D0_D1_D2": medias[0] < medias[1] < medias[2]
                })

    df_mono = pd.DataFrame(registros)
    if len(df_mono) == 0:
        return df_mono, pd.DataFrame()

    resumo = (
        df_mono
        .groupby(["Metodo", "Metrica"])["Monotonico_D0_D1_D2"]
        .mean()
        .mul(100)
        .reset_index()
        .rename(columns={"Monotonico_D0_D1_D2": "Percentual_monotonico_%"})
    )
    return df_mono, resumo


def selecionar_indice_por_dano_temperatura(df, falha, temperatura, ocorrencia=0):
    df_d = df[df["falha"] == falha].copy()
    if len(df_d) == 0:
        raise ValueError(f"Nenhuma curva encontrada para falha = {falha}")

    temps_disponiveis = np.array(sorted(df_d["temperatura_c"].unique()), dtype=float)
    temp_usada = temps_disponiveis[np.argmin(np.abs(temps_disponiveis - temperatura))]
    df_dt = df_d[np.isclose(df_d["temperatura_c"], temp_usada)].copy()

    if ocorrencia >= len(df_dt):
        ocorrencia = 0

    idx = df_dt.index[ocorrencia]
    return idx, temp_usada


def plot_curvas_ae_tres_danos(df_base, df_ae, df_hhat, y_ref_healthy,
                              healthy_by_temp, healthy_temps, fcols, fhz,
                              temperatura_escolhida=55, danos=(0, 1, 2),
                              ocorrencia=0, salvar=True, show=True):
    aplicar_estilo_artigo()
    fhz_khz = fhz / 1e3

    fig, axes = plt.subplots(len(danos), 1, figsize=(13, 5.5 * len(danos)), dpi=300, sharex=True)
    if len(danos) == 1:
        axes = [axes]

    for ax, dano in zip(axes, danos):
        idx_show, temp_usada = selecionar_indice_por_dano_temperatura(df_base, dano, temperatura_escolhida, ocorrencia)

        y_original = df_base.loc[idx_show, fcols].to_numpy(float)
        y_ae = df_ae.loc[idx_show, fcols].to_numpy(float)
        y_hhat = df_hhat.loc[idx_show, fcols].to_numpy(float)

        h_T, T_h_used = get_nearest_healthy_curve(healthy_by_temp, healthy_temps, temp_usada)

        assinatura_original = y_original - h_T
        assinatura_ae = y_ae - y_ref_healthy

        print("\n----------------------------------------------------")
        print(f"Dano {dano}")
        print(f"Índice usado: {idx_show}")
        print(f"Temperatura da curva: {temp_usada} °C")
        print(f"Saudável real mais próxima: {T_h_used} °C")
        print(f"Preservação RMSD residual = {rmsd(assinatura_ae, assinatura_original):.6f}")
        print(f"Preservação CCDM residual = {ccdm(assinatura_ae, assinatura_original):.6f}")

        ax.plot(fhz_khz, y_ref_healthy, "--", color="black", linewidth=1.5, label=f"Saudável REF {REF_TEMP} °C")
        ax.plot(fhz_khz, h_T, "-.", color="gray", linewidth=1.2, label=f"Saudável real {formatar_temp(T_h_used)} °C")
        ax.plot(fhz_khz, y_hhat, color="tab:green", linewidth=1.6, alpha=0.85, label="H_T estimado pelo AE")
        ax.plot(fhz_khz, y_original, color="tab:red", linewidth=1.1, alpha=0.75, label=f"Original D{dano} — {formatar_temp(temp_usada)} °C")
        ax.plot(fhz_khz, y_ae, color="tab:blue", linewidth=2.0, label="Compensada por AE térmico")

        ax.set_ylabel("Parte real da impedância")
        ax.set_title(f"Dano {dano} — {formatar_temp(temp_usada)} °C → REF saudável {REF_TEMP} °C")
        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.legend(frameon=True, facecolor="white", edgecolor="none")

    axes[-1].set_xlabel("Frequência (kHz)")
    plt.tight_layout()

    if salvar:
        salvar_figura(fig, f"Curvas_AE_termico_preserva_dano_Temp_{formatar_temp(temperatura_escolhida)}C")
    if show:
        plt.show()
    else:
        plt.close(fig)


def plot_ae_loss(history, salvar=True, show=True):
    aplicar_estilo_artigo()
    fig, ax = plt.subplots(figsize=(10, 5), dpi=300)
    ax.plot(history["epoch"], history["train_loss"], linewidth=2, label="Treino")
    ax.plot(history["epoch"], history["val_loss"], linewidth=2, label="Validação total")
    ax.plot(history["epoch"], history["val_recon"], linewidth=2, label="Validação reconstrução")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Treinamento do Autoencoder térmico saudável")
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=True, facecolor="white", edgecolor="none")
    plt.tight_layout()
    if salvar:
        salvar_figura(fig, "AE_loss_termico_saudavel")
    if show:
        plt.show()
    else:
        plt.close(fig)


def plot_histogramas(df_long, metricas=("RMSD", "CCDM"), bins=18,
                     nome_base="Histogramas_AE", salvar=True, show=True):
    aplicar_estilo_artigo()
    metodos = ["Original", "AE térmico"]
    danos = [0, 1, 2]
    cores = {"Original": "tab:gray", "AE térmico": "tab:blue"}
    alphas = {"Original": 0.35, "AE térmico": 0.65}

    fig, axes = plt.subplots(len(danos), len(metricas), figsize=(8.5 * len(metricas), 5.2 * len(danos)), dpi=300)
    if len(metricas) == 1:
        axes = np.array([[ax] for ax in axes])

    for i, dano in enumerate(danos):
        for j, metrica in enumerate(metricas):
            ax = axes[i, j]
            for metodo in metodos:
                vals = df_long.loc[(df_long["Metodo"] == metodo) & (df_long["falha"] == dano), metrica].dropna().to_numpy(float)
                if len(vals) == 0:
                    continue
                ax.hist(vals, bins=bins, alpha=alphas[metodo], color=cores[metodo], edgecolor="black", linewidth=0.7, label=metodo)
                ax.axvline(np.mean(vals), color=cores[metodo], linestyle="--", linewidth=2)
            ax.set_xlabel(metrica)
            ax.set_ylabel("Frequência")
            ax.set_title(f"{metrica} — Dano {dano}")
            ax.grid(False)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            if i == 0 and j == 0:
                ax.legend(frameon=True, facecolor="white", edgecolor="none")
    plt.tight_layout()
    if salvar:
        salvar_figura(fig, nome_base)
    if show:
        plt.show()
    else:
        plt.close(fig)


def avaliar_temperatura_auxiliar(df_comp):
    if "temperatura_pred_aux" not in df_comp.columns:
        return
    temp_true = df_comp["temperatura_c"].to_numpy(float)
    temp_pred = df_comp["temperatura_pred_aux"].to_numpy(float)
    mae = np.mean(np.abs(temp_true - temp_pred))
    rmse = np.sqrt(np.mean((temp_true - temp_pred) ** 2))
    print("\n==================== PREDIÇÃO AUXILIAR DE TEMPERATURA ====================")
    print("A temperatura NÃO foi input direto do AE; ela é estimada pelo latente.")
    print(f"MAE temperatura = {mae:.4f} °C")
    print(f"RMSE temperatura = {rmse:.4f} °C")


# ============================================================
# 8) EXECUÇÃO PRINCIPAL
# ============================================================

def executar_autoencoder_termico_saudavel():
    timings = {}

    print("====================================================")
    print("AUTOENCODER TÉRMICO SAUDÁVEL — PRESERVA DANO")
    print("====================================================")

    t0 = time.time()
    df = pd.read_pickle(ARQ_BASE).reset_index(drop=True)

    required_cols = {"temperatura_c", "falha"}
    missing = required_cols - set(df.columns)
    if len(missing) > 0:
        raise ValueError(f"Colunas obrigatórias ausentes: {missing}")

    fcols, fhz = get_freq_columns(df, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
    if len(fcols) == 0:
        raise ValueError("Nenhuma coluna de frequência encontrada na faixa escolhida.")

    healthy_by_temp, healthy_temps, y_ref_healthy, ref_temp_used = get_healthy_references_by_temperature(df, fcols, REF_TEMP)
    timings["load_reference"] = time.time() - t0

    print(f"\nTotal de amostras: {len(df)}")
    print(f"Classes de dano: {sorted(df['falha'].unique())}")
    print(f"Amostras saudáveis: {len(df[df['falha'] == 0])}")
    print(f"Faixa usada: {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
    print(f"Número de pontos: {len(fcols)}")
    print(f"Temperatura REF usada: {ref_temp_used} °C")

    # Original
    t0 = time.time()
    df_original_metricas = calcular_metricas_com_preservacao(
        df_curvas=df,
        df_original=df,
        fcols=fcols,
        y_ref_healthy=y_ref_healthy,
        healthy_by_temp=healthy_by_temp,
        healthy_temps=healthy_temps,
        metodo="Original"
    )
    timings["original_metrics"] = time.time() - t0

    # Autoencoder térmico
    t0 = time.time()
    ae_extra = train_autoencoder_saudavel(df, fcols, y_ref_healthy)
    df_ae_curvas, df_hhat_curvas, ae_latent = aplicar_autoencoder_compensacao(df, fcols, y_ref_healthy, ae_extra)

    df_ae_metricas = calcular_metricas_com_preservacao(
        df_curvas=df_ae_curvas,
        df_original=df,
        fcols=fcols,
        y_ref_healthy=y_ref_healthy,
        healthy_by_temp=healthy_by_temp,
        healthy_temps=healthy_temps,
        metodo="AE térmico"
    )
    timings["ae_termico"] = time.time() - t0

    # Juntar
    df_long = pd.concat([df_original_metricas, df_ae_metricas], axis=0, ignore_index=True)
    tabela_resumo = resumo_geral(df_long)
    tabela_temp = resumo_por_temperatura(df_long)
    df_mono, resumo_mono = checar_monotonicidade(df_long, metodos=("Original", "AE térmico"))

    # Salvar
    df_long.to_csv(os.path.join(OUTPUT_DIR, "df_long_metricas.csv"), index=False)
    tabela_temp.to_csv(os.path.join(OUTPUT_DIR, "resumo_por_temperatura.csv"), index=False)
    df_mono.to_csv(os.path.join(OUTPUT_DIR, "monotonicidade.csv"), index=False)
    resumo_mono.to_csv(os.path.join(OUTPUT_DIR, "resumo_monotonicidade.csv"), index=False)
    ae_extra["history"].to_csv(os.path.join(OUTPUT_DIR, "historico_loss_ae.csv"), index=False)
    pd.DataFrame(ae_latent).to_csv(os.path.join(OUTPUT_DIR, "latent_autoencoder.csv"), index=False)
    df_ae_curvas[["temperatura_c", "falha", "temperatura_pred_aux"]].to_csv(
        os.path.join(OUTPUT_DIR, "predicoes_auxiliares_ae.csv"), index=False
    )

    print("\n==================== RESUMO GERAL ====================")
    print(tabela_resumo)

    print("\n==================== MONOTONICIDADE ====================")
    print(resumo_mono)

    print("\n==================== TEMPOS ====================")
    for k, v in timings.items():
        print(f"{k:28s}: {v:.3f} s")

    avaliar_temperatura_auxiliar(df_ae_curvas)

    print("\n✅ Execução concluída.")

    return {
        "df_base": df,
        "df_ae_curvas": df_ae_curvas,
        "df_hhat_curvas": df_hhat_curvas,
        "df_original_metricas": df_original_metricas,
        "df_ae_metricas": df_ae_metricas,
        "df_long": df_long,
        "tabela_resumo": tabela_resumo,
        "tabela_temp": tabela_temp,
        "df_mono": df_mono,
        "resumo_mono": resumo_mono,
        "healthy_by_temp": healthy_by_temp,
        "healthy_temps": healthy_temps,
        "y_ref_healthy": y_ref_healthy,
        "ref_temp_used": ref_temp_used,
        "fcols": fcols,
        "fhz": fhz,
        "ae_extra": ae_extra,
        "ae_history": ae_extra["history"],
        "ae_latent": ae_latent,
        "timings": timings
    }


# ============================================================
# 9) RODAR TUDO
# ============================================================

resultados = executar_autoencoder_termico_saudavel()

df_base = resultados["df_base"]
df_ae_curvas = resultados["df_ae_curvas"]
df_hhat_curvas = resultados["df_hhat_curvas"]
df_long = resultados["df_long"]

healthy_by_temp = resultados["healthy_by_temp"]
healthy_temps = resultados["healthy_temps"]
y_ref_healthy = resultados["y_ref_healthy"]
fcols = resultados["fcols"]
fhz = resultados["fhz"]
ae_history = resultados["ae_history"]

tabela_resumo = resultados["tabela_resumo"]
tabela_temp = resultados["tabela_temp"]
df_mono = resultados["df_mono"]
resumo_mono = resultados["resumo_mono"]


# ============================================================
# 10) FIGURAS
# ============================================================

plot_curvas_ae_tres_danos(
    df_base=df_base,
    df_ae=df_ae_curvas,
    df_hhat=df_hhat_curvas,
    y_ref_healthy=y_ref_healthy,
    healthy_by_temp=healthy_by_temp,
    healthy_temps=healthy_temps,
    fcols=fcols,
    fhz=fhz,
    temperatura_escolhida=TEMP_ESCOLHIDA,
    danos=DANOS_PLOTAR,
    ocorrencia=OCORRENCIA_CURVA,
    salvar=True,
    show=True
)

plot_histogramas(
    df_long=df_long,
    metricas=("RMSD", "CCDM"),
    bins=HIST_BINS,
    nome_base="Histogramas_RMSD_CCDM_AE_termico",
    salvar=True,
    show=True
)

plot_histogramas(
    df_long=df_long,
    metricas=("DamageResidual_RMSD", "DamageResidual_CCDM"),
    bins=HIST_BINS,
    nome_base="Histogramas_Preservacao_Assinatura_AE_termico",
    salvar=True,
    show=True
)

plot_ae_loss(
    history=ae_history,
    salvar=True,
    show=True
)


# ============================================================
# 11) TABELAS
# ============================================================

print("\n==================== TABELA RESUMO ====================")
display(tabela_resumo)

print("\n==================== RESUMO POR TEMPERATURA ====================")
display(tabela_temp)

print("\n==================== MONOTONICIDADE POR TEMPERATURA ====================")
display(df_mono)

print("\n==================== RESUMO DA MONOTONICIDADE ====================")
display(resumo_mono)

print("\n✅ Todos os gráficos foram gerados e salvos.")
print(f"Pasta de saída: {OUTPUT_DIR}")
